# EMPIRE OF KINGS — Private 3D Studio (Kaggle free GPU)
This notebook starts the private Studio on a free Kaggle NVIDIA GPU session and exposes the same Studio UI through a temporary HTTPS tunnel.

Use a Kaggle Notebook with **GPU = NVIDIA P100** and **Internet = ON**. Kaggle currently documents free P100 access with a weekly GPU quota.

In [ ]:
# Check GPU first
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Activa un acelerador GPU NVIDIA en Session Options antes de continuar.')

In [ ]:
# Install the official TRELLIS stack and the private Studio dependencies.
%cd /kaggle/working
!rm -rf TRELLIS Empire-of-Kings
!git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git
!git clone --depth 1 https://github.com/nunezyenis05-beep/Empire-of-Kings.git
%cd /kaggle/working/TRELLIS
!bash -lc 'source /opt/conda/etc/profile.d/conda.sh && ./setup.sh --new-env --basic --xformers --diffoctreerast --spconv --mipgaussian --kaolin --nvdiffrast'
!bash -lc 'source /opt/conda/etc/profile.d/conda.sh && conda run -n trellis pip install fastapi uvicorn python-multipart pillow'

In [ ]:
# Start the Studio API/UI on the GPU machine.
import os, secrets, subprocess, time, pathlib
studio = pathlib.Path('/kaggle/working/Empire-of-Kings/aetherfall_project_2026_08_11/private_3d_studio')
data = pathlib.Path('/kaggle/working/eok_studio_data')
data.mkdir(parents=True, exist_ok=True)
token = secrets.token_urlsafe(32)
env = os.environ.copy()
env.update({
    'STUDIO_ADMIN_TOKEN': token,
    'STUDIO_DATA': str(data),
    'PYTHONPATH': '/kaggle/working/TRELLIS',
    'ATTN_BACKEND': 'xformers',
    'SPCONV_ALGO': 'native',
    'PORT': '7860',
})
log = open('/kaggle/working/eok_studio_server.log', 'w')
proc = subprocess.Popen(['bash','-lc', f'cd {studio} && exec /opt/conda/bin/conda run --no-capture-output -n trellis python server.py'], env=env, stdout=log, stderr=subprocess.STDOUT)
time.sleep(8)
print('Server PID:', proc.pid)
print('PRIVATE STUDIO TOKEN:', token)
print('Server log: /kaggle/working/eok_studio_server.log')

In [ ]:
# Create a free HTTPS Quick Tunnel.
%cd /kaggle/working
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
import subprocess, re, time
tunnel_log = open('/kaggle/working/cloudflared.log', 'w')
tunnel = subprocess.Popen(['./cloudflared','tunnel','--url','http://127.0.0.1:7860','--no-autoupdate'], stdout=tunnel_log, stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(2)
    text = open('/kaggle/working/cloudflared.log', errors='ignore').read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
    if m:
        url = m.group(0); break
print('STUDIO URL:', url or 'Tunnel todavía iniciando; revisa cloudflared.log')
print('TOKEN:', token)
print('Keep this Kaggle GPU session running while you create models.')